# ToyGPT Playground
All library code lives in the `toygpt` package (`src/toygpt/`); install it once with `pip install -e .` from the project root.
This notebook is for poking at things interactively. For full, reproducible runs use `experiments/train_gpt.py --config ...`.
`autoreload` picks up edits to the package without restarting the kernel.

In [ ]:
%load_ext autoreload
%autoreload 2

import torch
from toygpt.models import GPTConfig, ToyGPT
from toygpt.tokenization import BPETokenizer
from toygpt.data import load_wikitext, build_lm_datasets
from toygpt.training import TrainingConfig, train
from toygpt.inference import generate

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
train_corpus, val_corpus = load_wikitext()

## Our BPE tokenizer

In [ ]:
bpe_tokenizer = BPETokenizer(vocab_size=4000)  # You can adjust vocab_size
bpe_tokenizer.train(train_corpus)
ids = bpe_tokenizer.encode("the cat sat on the mat")
print(ids)
print(bpe_tokenizer.decode(ids))

## Train ToyGPT
Using the pretrained GPT-2 tokenizer for now. To use our BPE tokenizer instead, swap in `bpe_tokenizer`
and `GPTConfig(vocab_size=len(bpe_tokenizer.vocab))`.

In [ ]:
from transformers import GPT2Tokenizer
pretrained_tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

model_config = GPTConfig(vocab_size=pretrained_tokenizer.vocab_size)
model = ToyGPT(model_config)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"The model has {total_params:,} trainable parameters.")

train_dataset, val_dataset = build_lm_datasets(pretrained_tokenizer, train_corpus, val_corpus, model_config.max_seq_len)

In [ ]:
# num_workers=0: DataLoader worker processes are unreliable inside Jupyter on Windows
train_config = TrainingConfig(n_epochs=15, num_workers=0, out_dir="../runs/playground")
checkpoint = train(model, train_dataset, val_dataset, train_config, device)

## Generate

In [ ]:
model.load_state_dict(torch.load(checkpoint, map_location=device))
for prompt in ["The history of", "In the United States", "The game began"]:
    print(generate(model, pretrained_tokenizer, prompt, max_new_tokens=50))
    print()